In [4]:
import pandas as pd
import sqlite3

print("Libraries loaded successfully!")

df = pd.read_csv('/Users/carol/Desktop/ai_class_db/data/saleshourly.csv')
print(f"CSV loaded: {len(df)} rows, {len(df.columns)} columns")
print(df.head(3))

Libraries loaded successfully!
CSV loaded: 50532 rows, 13 columns
            datum  M01AB  M01AE  N02BA  N02BE  N05B  N05C  R03  R06  Year  \
0   1/2/2014 8:00    0.0   0.67    0.4    2.0   0.0   0.0  0.0  1.0  2014   
1   1/2/2014 9:00    0.0   0.00    1.0    0.0   2.0   0.0  0.0  0.0  2014   
2  1/2/2014 10:00    0.0   0.00    0.0    3.0   2.0   0.0  0.0  0.0  2014   

   Month  Hour Weekday Name  
0      1     8     Thursday  
1      1     9     Thursday  
2      1    10     Thursday  


In [5]:
print("=== DATA QUALITY CHECK ===")
print("\n1. Missing Values:")
print(df.isnull().sum())

print("\n2. Duplicate Rows:")
print(f"Total duplicates: {df.duplicated().sum()}")

print("\n3. Data Types:")
print(df.dtypes)

print("\n4. Basic Stats:")
print(df.describe())

print("\n5. Unique Weekdays:")
print(df['Weekday Name'].unique())

print("\n6. Zero sales rows (potential dead hours):")
drug_cols = ['M01AB','M01AE','N02BA','N02BE','N05B','N05C','R03','R06']
zero_rows = df[(df[drug_cols] == 0).all(axis=1)]
print(f"Rows where ALL drugs had zero sales: {len(zero_rows)}")

=== DATA QUALITY CHECK ===

1. Missing Values:
datum           0
M01AB           0
M01AE           0
N02BA           0
N02BE           0
N05B            0
N05C            0
R03             0
R06             0
Year            0
Month           0
Hour            0
Weekday Name    0
dtype: int64

2. Duplicate Rows:
Total duplicates: 0

3. Data Types:
datum               str
M01AB           float64
M01AE           float64
N02BA           float64
N02BE           float64
N05B            float64
N05C            float64
R03             float64
R06             float64
Year              int64
Month             int64
Hour              int64
Weekday Name        str
dtype: object

4. Basic Stats:
              M01AB         M01AE         N02BA         N02BE          N05B  \
count  50532.000000  50532.000000  50532.000000  50532.000000  50532.000000   
mean       0.209787      0.162365      0.161723      1.246842      0.368989   
std        0.556003      0.416109      0.453211      2.387392      0.9

In [6]:
print("=== DATA CLEANING & TRANSFORMATION ===")

# 1. Fix datum column - convert string to datetime
df['datum'] = pd.to_datetime(df['datum'])
print("✅ 1. datum converted to datetime:", df['datum'].dtype)

# 2. Min-Max Normalization for all drug sales columns
drug_cols = ['M01AB','M01AE','N02BA','N02BE','N05B','N05C','R03','R06']
for col in drug_cols:
    max_val = df[col].max()
    if max_val > 0:
        df[f'{col}_normalized'] = df[col] / max_val
    else:
        df[f'{col}_normalized'] = 0

print("✅ 2. All drug columns normalized using min-max normalization")
print("\nRaw vs Normalized comparison:")
print(df[drug_cols].max().rename('Raw Max'))
print(df[[f'{col}_normalized' for col in drug_cols]].max().rename('Normalized Max'))

=== DATA CLEANING & TRANSFORMATION ===
✅ 1. datum converted to datetime: datetime64[us]
✅ 2. All drug columns normalized using min-max normalization

Raw vs Normalized comparison:
M01AB     7.0
M01AE     6.0
N02BA     6.5
N02BE    29.0
N05B     15.0
N05C      6.0
R03      25.0
R06       5.0
Name: Raw Max, dtype: float64
M01AB_normalized    1.0
M01AE_normalized    1.0
N02BA_normalized    1.0
N02BE_normalized    1.0
N05B_normalized     1.0
N05C_normalized     1.0
R03_normalized      1.0
R06_normalized      1.0
Name: Normalized Max, dtype: float64


In [7]:
print("=== BUILDING DATABASE ===")

import sqlite3

# Connect to database 
conn = sqlite3.connect('/Users/carol/Desktop/ai_class_db/pharma.db')
cursor = conn.cursor()

print("✅ Connected to pharma.db")

# Create drugs table
cursor.execute('''
    CREATE TABLE IF NOT EXISTS drugs (
        drug_id INTEGER PRIMARY KEY AUTOINCREMENT,
        drug_code TEXT NOT NULL,
        drug_category TEXT NOT NULL
    )
''')
print("✅ drugs table created")

# Create time_periods table
cursor.execute('''
    CREATE TABLE IF NOT EXISTS time_periods (
        period_id INTEGER PRIMARY KEY AUTOINCREMENT,
        datum TEXT NOT NULL,
        year INTEGER,
        month INTEGER,
        hour INTEGER,
        weekday TEXT
    )
''')
print("✅ time_periods table created")

# Create sales table
cursor.execute('''
    CREATE TABLE IF NOT EXISTS sales (
        sale_id INTEGER PRIMARY KEY AUTOINCREMENT,
        period_id INTEGER,
        drug_id INTEGER,
        units_sold REAL,
        units_sold_normalized REAL,
        FOREIGN KEY (period_id) REFERENCES time_periods(period_id),
        FOREIGN KEY (drug_id) REFERENCES drugs(drug_id)
    )
''')
print("✅ sales table created")

conn.commit()
print("\n✅ All tables created successfully!")

=== BUILDING DATABASE ===
✅ Connected to pharma.db
✅ drugs table created
✅ time_periods table created
✅ sales table created

✅ All tables created successfully!


In [8]:
print("=== INSERTING DATA ===")

# 1. Insert drugs
drug_categories = {
    'M01AB': 'Anti-inflammatory',
    'M01AE': 'Anti-inflammatory',
    'N02BA': 'Analgesic',
    'N02BE': 'Analgesic',
    'N05B': 'Sedative',
    'N05C': 'Sleeping Pills',
    'R03': 'Respiratory',
    'R06': 'Antihistamine'
}

for code, category in drug_categories.items():
    cursor.execute('''
        INSERT INTO drugs (drug_code, drug_category) VALUES (?, ?)
    ''', (code, category))

conn.commit()
print("✅ Drugs inserted:", len(drug_categories))

# 2. Insert time_periods
for _, row in df.iterrows():
    cursor.execute('''
        INSERT INTO time_periods (datum, year, month, hour, weekday)
        VALUES (?, ?, ?, ?, ?)
    ''', (str(row['datum']), row['Year'], row['Month'], row['Hour'], row['Weekday Name']))

conn.commit()
print("✅ Time periods inserted:", len(df))

# 3. Insert sales
drug_codes = list(drug_categories.keys())
for period_id, row in enumerate(df.itertuples(), start=1):
    for drug_id, code in enumerate(drug_codes, start=1):
        units = getattr(row, code)
        units_norm = getattr(row, f'{code}_normalized')
        cursor.execute('''
            INSERT INTO sales (period_id, drug_id, units_sold, units_sold_normalized)
            VALUES (?, ?, ?, ?)
        ''', (period_id, drug_id, units, units_norm))

conn.commit()
print("✅ Sales records inserted:", len(df) * len(drug_codes))
print("\n✅ Database fully populated!")

=== INSERTING DATA ===
✅ Drugs inserted: 8
✅ Time periods inserted: 50532
✅ Sales records inserted: 404256

✅ Database fully populated!


In [9]:
print("=== VERIFYING DATABASE ===")

# Check drugs table
print("\n1. Drugs Table:")
result = cursor.execute('SELECT * FROM drugs').fetchall()
for row in result:
    print(row)

# Check time_periods table
print("\n2. Time Periods Table (first 5):")
result = cursor.execute('SELECT * FROM time_periods LIMIT 5').fetchall()
for row in result:
    print(row)

# Check sales table
print("\n3. Sales Table (first 5):")
result = cursor.execute('SELECT * FROM sales LIMIT 5').fetchall()
for row in result:
    print(row)

# Check total counts
print("\n4. Total Records:")
print("Drugs:", cursor.execute('SELECT COUNT(*) FROM drugs').fetchone()[0])
print("Time Periods:", cursor.execute('SELECT COUNT(*) FROM time_periods').fetchone()[0])
print("Sales:", cursor.execute('SELECT COUNT(*) FROM sales').fetchone()[0])

=== VERIFYING DATABASE ===

1. Drugs Table:
(1, 'M01AB', 'Anti-inflammatory')
(2, 'M01AE', 'Anti-inflammatory')
(3, 'N02BA', 'Analgesic')
(4, 'N02BE', 'Analgesic')
(5, 'N05B', 'Sedative')
(6, 'N05C', 'Sleeping Pills')
(7, 'R03', 'Respiratory')
(8, 'R06', 'Antihistamine')

2. Time Periods Table (first 5):
(1, '2014-01-02 08:00:00', 2014, 1, 8, 'Thursday')
(2, '2014-01-02 09:00:00', 2014, 1, 9, 'Thursday')
(3, '2014-01-02 10:00:00', 2014, 1, 10, 'Thursday')
(4, '2014-01-02 11:00:00', 2014, 1, 11, 'Thursday')
(5, '2014-01-02 12:00:00', 2014, 1, 12, 'Thursday')

3. Sales Table (first 5):
(1, 1, 1, 0.0, 0.0)
(2, 1, 2, 0.67, 0.11166666666666668)
(3, 1, 3, 0.4, 0.06153846153846154)
(4, 1, 4, 2.0, 0.06896551724137931)
(5, 1, 5, 0.0, 0.0)

4. Total Records:
Drugs: 8
Time Periods: 50532
Sales: 404256


In [10]:
print("=== ADDING INDEXES (Workshop 4) ===")

# Index on drug_id in sales table (frequent filter)
cursor.execute('CREATE INDEX IF NOT EXISTS idx_drug_id ON sales(drug_id)')
print("✅ Index created on sales.drug_id")

# Index on period_id in sales table (frequent join)
cursor.execute('CREATE INDEX IF NOT EXISTS idx_period_id ON sales(period_id)')
print("✅ Index created on sales.period_id")

# Index on year and month in time_periods (frequent filter)
cursor.execute('CREATE INDEX IF NOT EXISTS idx_year ON time_periods(year)')
print("✅ Index created on time_periods.year")

cursor.execute('CREATE INDEX IF NOT EXISTS idx_month ON time_periods(month)')
print("✅ Index created on time_periods.month")

cursor.execute('CREATE INDEX IF NOT EXISTS idx_hour ON time_periods(hour)')
print("✅ Index created on time_periods.hour")

conn.commit()
print("\n✅ All indexes created successfully!")

=== ADDING INDEXES (Workshop 4) ===
✅ Index created on sales.drug_id
✅ Index created on sales.period_id
✅ Index created on time_periods.year
✅ Index created on time_periods.month
✅ Index created on time_periods.hour

✅ All indexes created successfully!
